# Baseline Models — B0, B1, B2

Computes three lookup-table baselines against which all candidate models must be evaluated.

| ID | Model | Description | Purpose |
|----|-------|-------------|---------|
| B0 | Global mean | Predict mean `hours_creation` for every video | Absolute floor |
| B1 | Mean by length tier | Predict tier mean (Short / Average / Long) | **Success criterion** |
| B2 | Mean by video_type | Predict video_type mean | Diagnostic — expected to be noisy |

Means are computed from the **training set only**. No test labels are used.

In [3]:
import duckdb
import numpy as np
import pandas as pd
import sys

sys.path.append('..')
from src.constants import LENGTH_TIER_MAP, LENGTH_TIER_ORDER, LENGTH_TIER_ORDINAL

In [4]:
con = duckdb.connect('../data/workload.duckdb')
videos = con.execute('SELECT * FROM dim_videos_ml').df()
con.close()

videos_complete = videos[videos['is_complete'] == 1].reset_index(drop=True)
print(f"Total: {len(videos)} | Complete: {len(videos_complete)} | Excluded: {len(videos) - len(videos_complete)}")

Total: 126 | Complete: 98 | Excluded: 28


In [4]:
pd.set_option('display.max_rows', 100)

In [5]:
videos_complete.info()

<class 'pandas.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   video_id                        98 non-null     str           
 1   media_title                     98 non-null     str           
 2   video_type                      98 non-null     str           
 3   video_subtype                   98 non-null     str           
 4   media_type                      98 non-null     str           
 5   media_series                    98 non-null     str           
 6   total_hours                     98 non-null     float64       
 7   date_first                      98 non-null     datetime64[us]
 8   date_last                       98 non-null     datetime64[us]
 9   total_day_span                  98 non-null     int64         
 10  active_days_worked              98 non-null     int64         
 11  hours_creation     

## Train / Test Split

Ordered by `date_first` (when work began). Test set = 25 most recent videos. Baselines are computed from training-set means only.

In [5]:
N_TEST = 25


MODEL_COLS = [
    'expected_length_mins',
    'complexity_new',
    'complexity_media_depth',
    'complexity_delivery_style',
]

df_split = (
    videos_complete[
        ['date_first', 'hours_creation', 'media_title', 'video_type', 'media_type']
        + MODEL_COLS
    ]
    .dropna(subset=['hours_creation'])
    .sort_values('date_first')
    .reset_index(drop=True)
)

n_total     = len(df_split)
cutoff_idx  = n_total - N_TEST
cutoff_date = df_split.loc[cutoff_idx, 'date_first']

train_df = df_split.iloc[:cutoff_idx].copy()
test_df  = df_split.iloc[cutoff_idx:].copy()

print(f"Train: {len(train_df)}  ({train_df['date_first'].min().date()} → {train_df['date_first'].max().date()})")
print(f"Test:  {len(test_df)}   ({test_df['date_first'].min().date()} → {test_df['date_first'].max().date()})")
print(f"Cutoff: {cutoff_date.date()}")

Train: 73  (2023-02-13 → 2025-01-20)
Test:  25   (2025-02-06 → 2026-01-20)
Cutoff: 2025-02-06


## Feature Engineering

Three transformations applied to both train and test sets:

1. **`length_tier`** — maps raw `expected_length_mins` (7,10,15,20,25,30) to Short / Average / Long (for display and B1 grouping)
2. **`length_tier_ord`** — numeric ordinal encoding (Short=0, Average=1, Long=2) for Ridge regression
3. **`log_hours_creation`** — `log(hours_creation)` for the log-model variants (L1-log, L2-log)

`video_type` dummy encoding for Model B (L2) is handled inside the Ridge model notebook.

In [6]:
#1)Why does Ridge require numeric ordinal encoding?  Is this specific to Ridge regression?  Does tree-based, Lasso, etc
#   not need this?
#2) does the constants file even need "LENGTH_TIER_ORDER"?


def add_features(df):
    df = df.copy()

    # 1. Tier label (Short / Average / Long)
    df['length_tier'] = df['expected_length_mins'].astype('Int64').map(LENGTH_TIER_MAP)

    # 2. Ordinal encoding for Ridge (0=Short, 1=Average, 2=Long)
    df['length_tier_ord'] = df['length_tier'].map(LENGTH_TIER_ORDINAL)

    # 3. Log-transformed target for log-model variants
    df['log_hours_creation'] = np.log(df['hours_creation'])

    return df


train_df = add_features(train_df)
test_df  = add_features(test_df)

print("Feature engineering complete.")
print(f"\nLength tier distribution (train):")
print(train_df['length_tier'].value_counts().reindex(LENGTH_TIER_ORDER))
print(f"\nOrdinal encoding check:")
print(
    train_df[['expected_length_mins', 'length_tier', 'length_tier_ord']]
    .drop_duplicates()
    .sort_values('length_tier_ord')
    .to_string(index=False)
)
print(f"\nlog_hours_creation summary (train):")
print(train_df['log_hours_creation'].describe().round(3))

Feature engineering complete.

Length tier distribution (train):
length_tier
Short      38
Average    20
Long       15
Name: count, dtype: int64

Ordinal encoding check:
 expected_length_mins length_tier  length_tier_ord
                   10       Short                0
                    7       Short                0
                   15     Average                1
                   30        Long                2
                   20        Long                2
                   25        Long                2

log_hours_creation summary (train):
count    73.000
mean      2.672
std       0.555
min       1.552
25%       2.197
50%       2.659
75%       3.167
max       3.973
Name: log_hours_creation, dtype: float64


## Baselines

Lookup tables computed from **training set only**. RMSE evaluated on the 25-video test set.

np.float64(16.80712328767123)

In [8]:



def rmse(actual, predicted):
    return np.sqrt(np.mean((predicted - actual) ** 2))
#RMSE = sqrt( (1/n) * sum( (y_i - ŷ_i)² ) )
#taking the mean of that expression inside the sqrt in above code is shorthand way of doing this

# ── B0: Global mean ──────────────────────────────────────────────────────────
b0_mean = train_df['hours_creation'].mean()

test_df['b0_pred'] = b0_mean
b0_rmse = rmse(test_df['hours_creation'], test_df['b0_pred'])

print(f"B0 — Global mean: {b0_mean:.2f}h  →  Test RMSE: {b0_rmse:.2f}h")


# ── B1: Mean by length tier ───────────────────────────────────────────────────
b1_lookup = train_df.groupby('length_tier')['hours_creation'].mean()

test_df['b1_pred'] = test_df['length_tier'].map(b1_lookup)
b1_rmse = rmse(test_df['hours_creation'], test_df['b1_pred'])

print(f"\nB1 — Mean by length tier (training means):")
print(b1_lookup.reindex(LENGTH_TIER_ORDER).round(2).to_string())
print(f"\nB1 Test RMSE: {b1_rmse:.2f}h")


# ── B2: Mean by video_type ────────────────────────────────────────────────────
b2_lookup = train_df.groupby('video_type')['hours_creation'].mean()

test_df['b2_pred'] = test_df['video_type'].map(b2_lookup)
b2_rmse = rmse(test_df['hours_creation'], test_df['b2_pred'])

print(f"\nB2 — Mean by video_type (training means):")
print(b2_lookup.sort_values().round(2).to_string())
print(f"\nB2 Test RMSE: {b2_rmse:.2f}h")

B0 — Global mean: 16.81h  →  Test RMSE: 6.50h

B1 — Mean by length tier (training means):
length_tier
Short      10.48
Average    24.00
Long       23.25

B1 Test RMSE: 5.45h

B2 — Mean by video_type (training means):
video_type
Rankings       14.12
Review         16.79
Playthrough    20.55

B2 Test RMSE: 6.80h


## Summary

| ID | Model | Test RMSE | Notes |
|----|-------|-----------|-------|
| B0 | Global mean | — | Absolute floor |
| B1 | Mean by length tier | — | **Success criterion** |
| B2 | Mean by video_type | — | Diagnostic |

*(Run the cell above to populate)*

In [9]:
print("=" * 45)
print(f"  B0 (global mean)       RMSE: {b0_rmse:.2f}h")
print(f"  B1 (by length tier)    RMSE: {b1_rmse:.2f}h  ← success criterion")
print(f"  B2 (by video_type)     RMSE: {b2_rmse:.2f}h")
print("=" * 45)
print(f"  B1 lift over B0: {b0_rmse - b1_rmse:.2f}h")
print(f"  Target threshold: ±5h RMSE to be useful")

  B0 (global mean)       RMSE: 6.50h
  B1 (by length tier)    RMSE: 5.45h  ← success criterion
  B2 (by video_type)     RMSE: 6.80h
  B1 lift over B0: 1.05h
  Target threshold: ±5h RMSE to be useful
